# **Build a Mini-BERT Model From Scratch**

In [61]:
#Import Libraries

import torch
import torch.nn as nn 
from tqdm import tqdm
from transformers import BertTokenizer
import requests
from zipfile import ZipFile
from torch.utils.data import Dataset
import pandas as pd
import json
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tensorflow import Tensor
import math
import ast
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


In [5]:
print(torch.__version__)

2.11.0+cpu


In [7]:
#define tokenizor
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
# def download (url, filename):
#     response = requests.get(url)
#     if response.status_code == 200:
#         with open(filename, 'wb') as f:
#             f.write(response.content)
#     else:
#         print(f"Failed to download. Status code: {response.status_code}")

In [9]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bZaoQD52DcMpE7-kxwAG8A.zip'
filename = 'BERT-dataset'

In [10]:
# download(url, filename)

In [11]:
# def unzip_file (filename,directory):
#     with ZipFile(filename, 'r') as unzip_f:
#         unzip_f.extractall(directory)

In [12]:
# directory = 'D:\Projects\mini-bert-nsp-mlm-pretraining'

In [13]:
# unzip_file(filename,directory)

In [14]:
data = pd.read_csv('bert_dataset/bert_train_data.csv')
data.iloc[2]

Original Text    I rented I AM CURIOUS-YELLOW from my video sto...
BERT Input       [1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...
BERT Label       [0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...
Segment Label    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...
Is Next                                                          1
Name: 2, dtype: object

In [15]:
row =  data.iloc[2]
ans = torch.tensor(ast.literal_eval(row["BERT Input"]))
print(ans.type)

<built-in method type of Tensor object at 0x00000143F39C2F30>


In [16]:
data.head()

,Original Text,BERT Input,BERT Label,Segment Label,Is Next
0,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 94, 12615, 5, 1026, 22, 5, 201, 18, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
1,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 1571, 16, 249, 35471, 3, 67, 3, 1138, ...","[0, 0, 0, 0, 0, 0, 46, 0, 401, 0, 0, 0, 0, 5, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
2,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...","[0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
3,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 9266, 54, 14, 3, 783, 11, 2483, 17, 3, 7, ...","[0, 0, 0, 0, 134, 0, 0, 0, 0, 685, 7, 0, 0, 9,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
4,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 66, 14519, 4444, 7, 4637, 76, 1479, 11, 60...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 60, 0, 0, 7552, 0,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1


### **Build Dataset Class**

In [17]:
class BertDatatsetCSV(Dataset):
    def __init__(self, filename):

        self.dataset = pd.read_csv(filename)
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def __len__(self):
        return len(self.dataset)

    def safe_json(self, x):
        if isinstance(x, str):
            return json.loads(x)
        return x

    def __getitem__(self, idx):
        self.dataset = self.dataset.reset_index(drop=True)
        row = self.dataset.iloc[idx]

        bert_input = torch.tensor(
            ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
        )

        bert_label = torch.tensor(
            ast.literal_eval(str(row["BERT Label"]).strip()), dtype=torch.long
        )

        segment_label = torch.tensor(
            ast.literal_eval(str(row["Segment Label"]).strip()), dtype=torch.long
        )

        is_next = torch.tensor(row["Is Next"], dtype=torch.long)

        return bert_input, bert_label, segment_label, is_next

In [18]:
row =  data.iloc[31360]
bert_input = torch.tensor(json.loads(row['BERT Input']), dtype=torch.long)
bert_input

tensor([   1,   49,   12,   19,   47,   84,  365,  135,    6,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,   21, 1040,
           7,    5,    3,    3,   94,  135,    3, 1142,    3,   22,  294, 8429,
           7,    3,  143,    3,   34,   32,    3,    2])

In [19]:
data = data.reset_index(drop=True)
row = data.iloc[3]

bert_input = torch.tensor(
        ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
    )
bert_input

tensor([    1,  9266,    54,    14,     3,   783,    11,  2483,    17,     3,
            7,  1578,   121,     3,   345,    10,   117,  1163,     3,    16,
           75,    78,     3,    77,     3,    22,   540,     6,     2,     0,
           15,   206,  2185,  7274,     3,  1922,     3,    10, 21481,    53,
            3,  4659,    31,  2384,     7,    64,    55,   405,    23,    50,
          477,  1695,     7,  8138,     7,     8,  1002,     3,     6,     2])

### **Build Collate Function**

In [20]:
PAD_IDX = 0

def collate_func(batch):
    bert_input_batch = []
    bert_label_batch = []
    segment_label_batch = []
    is_next_batch = []

    for bert_input, bert_label, segment_label, is_next in batch:
        bert_input_batch.append(bert_input)
        bert_label_batch.append(bert_label)
        segment_label_batch.append(segment_label)
        is_next_batch.append(is_next)

    bert_input_batch = pad_sequence(
        bert_input_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    bert_label_batch = pad_sequence(
        bert_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    segment_label_batch = pad_sequence(
        segment_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    is_next_batch = torch.tensor(is_next_batch, dtype=torch.long)

    return bert_input_batch, bert_label_batch, segment_label_batch, is_next_batch

### **Initialize the Dataset and Dataloaders**

In [21]:
train_dataset = BertDatatsetCSV("./bert_dataset/bert_train_data.csv")
test_dataset = BertDatatsetCSV("./bert_dataset/bert_test_data.csv")

In [22]:
train_dataset.__len__()

170540

In [23]:
test_dataset.__len__()

167449

In [24]:
batch_size = 4

In [25]:
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, collate_fn=collate_func,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size, collate_fn=collate_func)

### **Model Implementation**

Model Creation (BERT Embeddings)

In BERT, three types of embeddings are combined to represent each input token:

1. Token Embedding

This is the basic representation of each word/token.

* Maps every token to a fixed-size dense vector
* Learns semantic meaning of words
* Captures relationships between tokens based on context

👉 Think of it as: “What does this word mean?”

2. Positional Embedding

Transformers process tokens in parallel, so they don’t naturally understand order.

* Adds position information to each token
* Encodes where a token appears in the sequence
* Helps the model understand word order and structure

👉 Think of it as: “Where is this word in the sentence?”

3. Segment Embedding

Used when BERT processes multiple sentences (e.g., sentence pairs).

* Assigns different embeddings to different segments (Sentence A, Sentence B)
* Helps distinguish between parts of the input
* Useful for tasks like question answering and next sentence prediction

👉 Think of it as: “Which sentence does this word belong to?”

Final Input Representation

The final embedding for each token is the sum of all three:

* Final Embedding = Token Embedding + Positional Embedding + Segment Embedding

This combined representation allows BERT to understand meaning, order, and context simultaneously.

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

#### **Token Embeddings**

In [27]:
class TokenEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, num_dims):
        super().__init__()
        self.num_dims = num_dims
        self.embeddings = nn.Embedding(vocab_size, num_dims)
    
    def forward(self, input_tokens:Tensor):
        embedding_out = self.embeddings(input_tokens.long())
        return embedding_out * math.sqrt(self.num_dims)

#### **Position Embedding**

In [28]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        positions = torch.arange(0, max_seq_len).unsqueeze(1)

        pe = torch.zeros((max_seq_len, d_model))

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)

        pe = pe.unsqueeze(1)
        self.register_buffer("positional_encodings", pe)

    def forward(self, word_embeddings):
        # word_embeddings -> [seq_len, batch_size, num_dims]
        seq_len = word_embeddings.size(0)
        positional_embeddings = (
            word_embeddings + self.positional_encodings[0:seq_len, :, :]
        )
        return self.dropout(positional_embeddings)

#### **BERT Embeddings**

In [29]:
class BERTEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, d_models,max_seq_len, dropout):
        super().__init__()
        self.token_embeddings = TokenEmbeddings(vocab_size, d_models)
        self.positional_embeddings = PositionalEmbeddings(max_seq_len,d_models,dropout)
        self.segment_embeddings = nn.Embedding(3,d_models)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self,word_tokens, segment_ids):
        embed_out = self.token_embeddings(word_tokens)
        pos_embed = self.positional_embeddings(embed_out)
        segment_embed = self.segment_embeddings(segment_ids)
        
        bert_embeddings = pos_embed + segment_embed
        return self.dropout(bert_embeddings)

#### **Initiate Sample Dataset for Testing**

In [30]:
count = 0

for batch in train_dataloader:
    bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
    
    count += 1
    if count == 5:
        break

### **BERT Model Architecture Overview**

* The complete BERT model consists of the following key components:

1. Initialization

The BERT class extends torch.nn.Module. During initialization, it defines core hyperparameters such as:

* Vocabulary size
* Hidden/model dimension
* Number of encoder layers
* Number of attention heads
* Dropout rate

These parameters configure the overall architecture of the model.

2. Embedding Layer

The embedding layer is implemented using the BERTEmbedding class. It combines:

* Token embeddings
* Segment embeddings

This layer converts input tokens into dense vector representations suitable for the transformer.

3. Transformer Encoder

A stack of Transformer Encoder layers processes the embeddings. Each layer applies:

* Multi-head self-attention
* Feedforward neural networks

The depth (number of layers), number of attention heads, model dimension, and dropout are controlled by the initialized parameters.

4. Next Sentence Prediction (NSP) Head

A linear classification layer is used for the NSP task. It:

* Takes the encoder output (typically the [CLS] token representation)
* Predicts whether two sentences are consecutive
* Outputs probabilities for two classes (IsNext / NotNext)

5. Masked Language Modeling (MLM) Head

Another linear layer handles the MLM task. It:

* Uses encoder outputs for all token positions
* Predicts the original tokens for masked positions
* Produces probability distributions over the vocabulary

6. Forward Pass

The forward method defines how data flows through the model:

* Inputs: bert_inputs, segment_labels
* Process: Embedding → Transformer Encoder → Task-specific heads

Outputs:
* NSP predictions
* MLM predictions

In [31]:
class BERTModel (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, n_layers, n_heads, dropout = 0.1):
        
        super().__init__()
        #define bert embedding
        self.bert_embedding = BERTEmbeddings(vocab_size,d_model,max_seq_len,dropout)
        # define encoder layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, 
                                                        dropout=dropout, batch_first=False)
        # define encoder
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=n_layers)
        
        #define linear layer for mask language modeling
        self.fc_mlm = nn.Linear(d_model, vocab_size)
        
        #define linear layer for next word prediction
        self.fc_nsp = nn.Linear(d_model, 2)
        
    def forward(self,bert_inputs, segment_labels):
        bert_out = self.bert_embedding(bert_inputs, segment_labels)
        
        #we need to create padding mask to prevent attention to padding
        pad_mask = (bert_inputs == PAD_IDX).transpose(0,1)
        
        encoder_out = self.encoder(bert_out, src_key_padding_mask = pad_mask)
        
        # for mlm
        mlm_out = self.fc_mlm(encoder_out)
        
        #for nsp
        nsp_out = self.fc_nsp(encoder_out[0,:,:])
        
        return mlm_out, nsp_out

#### **Create a Instance of the Model**

In [32]:
vocab_size = 147161
d_model = 12
n_layers = 2
nheads = 4
dropout = 0.1

In [33]:
bert_model = BERTModel(vocab_size,d_model,1000,n_layers,nheads,dropout)

C:\Users\Vish\AppData\Local\Temp\ipykernel_1544\3945956926.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=n_layers)


In [34]:
padding_mask = (bert_inputs == PAD_IDX).transpose(0,1)
padding_mask.shape

torch.Size([4, 62])

In [35]:
bert_inputs.shape

torch.Size([62, 4])

In [36]:
bert_inputs[:,0]

tensor([   1,   64,  651, 2421,    8,  781,   23,   17,    3,    2,   25,   64,
          61,   12,    3,    3,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0])

In [37]:
token_embddings = TokenEmbeddings(vocab_size,d_model)

t_embeddings = token_embddings(bert_inputs)
t_embeddings.shape


torch.Size([62, 4, 12])

In [38]:
positional_encoding = PositionalEmbeddings(max_seq_len=500,d_model=d_model,dropout=dropout)
pe = positional_encoding(t_embeddings)
pe.shape    

torch.Size([62, 4, 12])

In [39]:
for i in range(3):
    print(f"postional embddings  for the {i}th token of the first sample: {pe[i,0,:]}")

postional embddings  for the 0th token of the first sample: tensor([-5.1482,  4.1859,  0.2579, 12.1042,  1.8224,  2.4809,  0.9969, -2.0607,
        -0.2282,  4.5214,  3.4019, -5.6426], grad_fn=<SelectBackward0>)
postional embddings  for the 1th token of the first sample: tensor([-2.6748,  0.1139,  2.2644,  0.7140, -0.0000, -3.3152,  7.3287, -0.0000,
        -1.6711,  0.0000,  0.2755,  0.0000], grad_fn=<SelectBackward0>)
postional embddings  for the 2th token of the first sample: tensor([-1.0184, -4.2672, -0.8736,  4.8702, -2.4050, -5.2163,  4.1704,  0.0000,
        -1.1425,  0.0000, -4.1301,  9.6450], grad_fn=<SelectBackward0>)


In [40]:
segment_embedding = nn.Embedding(3,d_model)
se = segment_embedding(segment_labels)
se.shape

torch.Size([62, 4, 12])

In [41]:
for i in range(3):
    print(f"segment embddings  for the {i}th token of the first sample: {se[i,0,:]}")

segment embddings  for the 0th token of the first sample: tensor([ 1.1972, -0.0745, -1.4661, -2.0005,  1.4036, -0.6115, -0.0584, -1.2230,
        -0.1772, -0.0411,  2.0243, -0.3066], grad_fn=<SelectBackward0>)
segment embddings  for the 1th token of the first sample: tensor([ 1.1972, -0.0745, -1.4661, -2.0005,  1.4036, -0.6115, -0.0584, -1.2230,
        -0.1772, -0.0411,  2.0243, -0.3066], grad_fn=<SelectBackward0>)
segment embddings  for the 2th token of the first sample: tensor([ 1.1972, -0.0745, -1.4661, -2.0005,  1.4036, -0.6115, -0.0584, -1.2230,
        -0.1772, -0.0411,  2.0243, -0.3066], grad_fn=<SelectBackward0>)


In [42]:
bert_embeddings = pe + se
bert_embeddings.shape

torch.Size([62, 4, 12])

In [43]:
for i in range(3):
    print(f"bert embddings  for the {i}th token of the first sample: {bert_embeddings[i,0,:]}")

bert embddings  for the 0th token of the first sample: tensor([-3.9510,  4.1114, -1.2082, 10.1037,  3.2260,  1.8695,  0.9385, -3.2837,
        -0.4054,  4.4802,  5.4262, -5.9491], grad_fn=<SelectBackward0>)
bert embddings  for the 1th token of the first sample: tensor([-1.4776,  0.0395,  0.7983, -1.2865,  1.4036, -3.9267,  7.2703, -1.2230,
        -1.8483, -0.0411,  2.2998, -0.3066], grad_fn=<SelectBackward0>)
bert embddings  for the 2th token of the first sample: tensor([ 0.1788, -4.3416, -2.3397,  2.8697, -1.0014, -5.8278,  4.1119, -1.2230,
        -1.3197, -0.0411, -2.1058,  9.3385], grad_fn=<SelectBackward0>)


#### **Idea About the Encdoing Layers (Optional)**

In [44]:
encoding_layer = nn.TransformerEncoderLayer(d_model,nheads,dropout=dropout, batch_first=False)
transformer_encoder = nn.TransformerEncoder(encoder_layer=encoding_layer, num_layers=n_layers)

#pass the bert embeddings to transformer encoder
te = transformer_encoder(bert_embeddings, src_key_padding_mask = padding_mask)
te.shape

C:\Users\Vish\AppData\Local\Temp\ipykernel_1544\3737590608.py:2: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  transformer_encoder = nn.TransformerEncoder(encoder_layer=encoding_layer, num_layers=n_layers)


torch.Size([62, 4, 12])

In [45]:
for i in range(3):
    print(f"transformer encoder  for the {i}th token of the first sample: {te[i,0,:]}")

transformer encoder  for the 0th token of the first sample: tensor([-0.3623,  0.4656, -0.4442,  1.3664,  1.1014,  0.2068,  0.1141, -1.8220,
        -0.6026,  0.4342,  1.2449, -1.7022], grad_fn=<SelectBackward0>)
transformer encoder  for the 1th token of the first sample: tensor([-0.0827, -0.6781, -0.4417, -1.7460, -0.4980,  0.3736,  1.8534,  0.4251,
        -0.4249, -0.4774,  1.9525, -0.2558], grad_fn=<SelectBackward0>)
transformer encoder  for the 2th token of the first sample: tensor([ 0.5770, -1.4817, -1.5044,  0.5129, -0.2932, -0.7353,  1.4705, -0.3388,
        -0.3696, -0.4640,  0.9673,  1.6594], grad_fn=<SelectBackward0>)


##### **Next Sentence Prediction**

In [46]:
nsp_layer = nn.Linear(d_model,2)
nsp = nsp_layer(te[0,:])
nsp.shape

torch.Size([4, 2])

In [47]:

print(nsp)

tensor([[-0.5282, -0.1330],
        [-0.8348, -0.2767],
        [-0.6422, -0.3538],
        [-0.5619, -0.1967]], grad_fn=<AddmmBackward0>)


##### **Mask Language Modeling**

In [48]:
mlm_layer = nn.Linear(d_model, vocab_size)
mlm = mlm_layer(te)

mlm.shape

torch.Size([62, 4, 147161])

In [49]:
print(mlm[0:3,:,:])

tensor([[[-3.8200e-01, -5.0065e-02, -4.7686e-01,  ...,  4.0689e-01,
          -3.3149e-04, -1.3062e+00],
         [-8.6189e-01, -5.4021e-02, -2.1128e-01,  ...,  2.5811e-01,
           4.8410e-01, -1.1618e+00],
         [ 4.0884e-01,  7.8475e-01, -2.4475e-01,  ...,  1.2549e+00,
          -4.8596e-01, -3.5527e-01],
         [-3.5506e-01,  4.0389e-02, -4.7150e-01,  ...,  2.4157e-01,
           4.2982e-02, -1.0117e+00]],

        [[ 2.1457e-01,  8.9488e-01,  1.5962e-01,  ...,  6.8786e-01,
          -1.7781e-01,  6.9146e-01],
         [-1.1068e+00, -5.6199e-01,  8.8962e-01,  ..., -8.4406e-01,
           8.8852e-01, -1.1242e+00],
         [-5.4759e-02,  2.5679e-01,  5.7341e-01,  ..., -7.8698e-01,
          -1.0706e-01,  6.0449e-01],
         [-9.8151e-01,  2.3421e-01,  3.2430e-01,  ...,  3.3228e-01,
           7.1339e-01,  2.1219e-01]],

        [[-6.8310e-01,  2.3532e-01,  1.2448e-01,  ..., -2.2443e-01,
           1.7803e-01,  7.0699e-01],
         [ 8.5857e-01,  7.7761e-01, -6.5540e-01,  .

In [50]:
bert_model = bert_model.to(device)

## **Model Evaluation, Training and Prediction**

In [51]:
# The loss function must ignore PAD tokens and only calculates loss for the masked tokens
mlm_loss = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
nsp_loss = nn.CrossEntropyLoss()

### **Model Evaluation** 

In [52]:
def model_eval(model:BERTModel, dataloader:DataLoader, nsp_loss = nsp_loss, mlm_loss = mlm_loss, device=device):
    
    model.eval()
    
    nsp_loss_val = 0
    mlm_loss_val = 0
    total_batches = 0
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
            
            #predict the values from the model
            mlm_out_predict, nsp_out_predict = model(bert_inputs, segment_labels)
            
            # calculate loss for each prediction
            mlm_loss_batch = mlm_loss(mlm_out_predict.view(-1,mlm_out_predict.size(-1)),bert_labels.view(-1))
            
            nsp_loss_batch = nsp_loss(nsp_out_predict, is_nexts.view(-1))
            
            batch_loss = mlm_loss_batch + nsp_loss_batch
            
            if torch.isnan(batch_loss):
                continue
            else:
                total_loss += batch_loss.item()
                nsp_loss_val += nsp_loss_batch.item()
                mlm_loss_val += mlm_loss_batch.item()
                
                total_batches += 1
    
    avg_loss = total_loss / (total_batches )
    avg_nsp_loss = nsp_loss_val / (total_batches )
    avg_mlm_loss = mlm_loss_val / (total_batches)
    
    print(f"Average Loss: {avg_loss:.4f}, Average Next Sentence Loss: {avg_nsp_loss:.4f}, Average Mask Loss: {avg_mlm_loss:.4f}")
    return avg_loss

Hence our dataset is huge it may takes several hours to train the model. because of that we use sample dataset to train the model

In [53]:
train_sample_path = "./bert_dataset/bert_train_data_sampled.csv"
test_sample_path = "./bert_dataset/bert_test_data_sampled.csv"

train_sample_dataset = BertDatatsetCSV(train_sample_path)
test_sample_dataset = BertDatatsetCSV(test_sample_path)

In [54]:
train_sample_dataloader = DataLoader(train_sample_dataset,batch_size=batch_size,
                                    shuffle=True, collate_fn=collate_func)

test_sample_dataloader = DataLoader(test_sample_dataset,batch_size=batch_size, 
                                    shuffle=False, collate_fn=collate_func)


#### Average Losses Before Model Training

In [55]:
# average_loss = model_eval(bert_model,test_sample_dataloader)
# average_loss

In [56]:
# print(average_loss)

### **Model Training**


In [57]:
optimizer = AdamW(bert_model.parameters(),lr=1e-4, weight_decay=0.01, betas=(0.9,0.999))

In [59]:
num_epochs = 1
total_step = num_epochs * len(train_sample_dataloader)
warm_steps = int(total_step * 0.1)

scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=warm_steps,
                                            num_training_steps=total_step)

In [ ]:
def model_train(model:BERTModel, dataloader:DataLoader, nsp_loss=nsp_loss,
                mlm_loss = mlm_loss, optimizer = optimizer, lr_scehduler = scheduler,
                device = device):
    
    train_loss = []
    eval_loss = []
    
    for epoch in tqdm(range(num_epochs),desc="Training Epochs"):
        
        model.train()
        epoch_loss = 0
        
        for step_id, batch in enumerate(tqdm(dataloader,desc=f"epoch {epoch+1}")):
            bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
    